<a href="https://colab.research.google.com/github/jenny4890/deepLearning/blob/main/CNN123.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import torchvision
import matplotlib.pyplot as plt

if torch.backends.mps.is_available():
    my_device = torch.device('mps')
elif torch.cuda.is_available():
    my_device = torch.device('cuda')
else:
    my_device = torch.device('cpu')

print(my_device)


cuda


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([ #여러 변환을 한 번에 적용하는 “파이프라인”
    transforms.ToTensor(), #이미지(PIL 이미지나 numpy 배열)를PyTorch가 학습할 수 있는 Tensor 형태로 바꿈
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #RGB 평균, 분산 after this data range = -1~1
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
#train=True -> trainset
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
#train=False -> testset
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:05<00:00, 32.1MB/s]


In [ ]:
class mordernGAPCNN(nn.Module):
  def __init__(self,  num_classes=10):
    super().__init__()
    #아래 batchnorm은 항상 Conv 다음에 해 준다 Relu가 있든 없든. 그래서 data를 안정화 시킨다.
    self.features = nn.Sequential(
       nn.Conv2d(3,64, kernel_size = 3, stride = 1, padding=1, bias= False), #3 input channels, 64 output channel
       nn.BatchNorm2d(64), # input channel
       nn.ReLU(inplace=True), # 기존 입력텐서가 위치한 메모리 주소에 결과값을 Overwrite
       nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
       nn.BatchNorm2d(128),
       nn.ReLU(inplace=True)
    )

    self.gap = nn.AdaptiveAvgPool2d(1)
    self.classifier = nn.Linear(128, num_classes)

  def forward(self, x):
      x = self.features(x)
      x = self.gap(x)
      x = torch.flatten(x,1)
      x = self.classifier(x)
      return x

      return x
# Create the model
net = mordernGAPCNN(num_classes=10)


In [ ]:
lossfn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr = 0.0001)


In [ ]:
net.to(my_device)
num_epochs=100
for epoch in range(num_epochs):
  net.train()
  for batchidx,(data,label) in enumerate(trainloader):
    data, label = data.to(my_device), label.to(my_device)
    output = net(data)
    loss = lossfn(output, label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  net.eval() # dropout(모두 ON) or batchnorm(running avg, variance)을 eval 모드로 동작시켜.
  val_loss=0.0
  correct=0
  with torch.no_grad(): #미분결과 저장하지마 라는 명령 메모리 효육
    for data, label in testloader:
      data, label = data.to(my_device), label.to(my_device)
      output = net(data)
      loss = lossfn(output, label) #error 평균
      val_loss +=loss.item()*data.size(0) # (N, C, H, W)중에 0번째 그러니까 batchsize.
      #loss.item()는 torch.Tensor to 	float변환

      predicted = output.argmax(dim=1) # dim=0 batch 방향, dim=1 가로 방향 class방향.
      #이거 이해 해야함! 만약 배치 사이즈가 3이고 클래스가 4개인 간단한 예시가 있다고 가정해 봅시다. output의 형태는 (3, 4)가 됩니다.
      '''
      (3, 4) 3이 0, 4가 1 dim=0 (세로 방향 / Batch 방향): dim=1 (가로 방향 / Class 방향):
      # [이미지1의 점수 4개], [이미지2의 점수 4개], [이미지3의 점수 4개]
      output = torch.tensor([
      [0.1, 0.2, 0.7, 0.0],  # 이미지 0 (정답 후보들)
      [0.1, 0.9, 0.0, 0.0],  # 이미지 1 (정답 후보들)
      [0.3, 0.3, 0.2, 0.2]   # 이미지 2 (정답 후보들)
      )
      '''
      correct += predicted.eq(label).sum().item() #item()는 torch.Tensor를float변환


  val_loss /= len(testloader.dataset) # 모델 로스
  val_accuracy = 100. * correct / len(testloader.dataset) # 실제 데이타 맞출확률
  print(f"Epoch [{epoch + 1}/{num_epochs}], Training Loss: {loss.item():.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")



Epoch [1/100], Training Loss: 1.5898, Validation Loss: 1.6449, Validation Accuracy: 42.66%
Epoch [2/100], Training Loss: 1.2711, Validation Loss: 1.5307, Validation Accuracy: 46.21%
Epoch [3/100], Training Loss: 1.3344, Validation Loss: 1.4718, Validation Accuracy: 48.61%
Epoch [4/100], Training Loss: 1.2058, Validation Loss: 1.4610, Validation Accuracy: 48.87%
Epoch [5/100], Training Loss: 1.0997, Validation Loss: 1.3820, Validation Accuracy: 51.46%
Epoch [6/100], Training Loss: 1.0839, Validation Loss: 1.3814, Validation Accuracy: 51.67%
Epoch [7/100], Training Loss: 1.0626, Validation Loss: 1.3529, Validation Accuracy: 52.77%
Epoch [8/100], Training Loss: 1.1132, Validation Loss: 1.3347, Validation Accuracy: 53.25%
Epoch [9/100], Training Loss: 1.0553, Validation Loss: 1.3257, Validation Accuracy: 53.67%
Epoch [10/100], Training Loss: 1.0428, Validation Loss: 1.2978, Validation Accuracy: 54.74%
Epoch [11/100], Training Loss: 0.7657, Validation Loss: 1.2805, Validation Accuracy: 54.9